# Colab ComfyUI(简版)
在 Colab 上直接运行一个标准 ComfyUI,在浏览器里完成全部出图/重绘流程。不需要远程插件、不需要本机配置。

- 模型读取 `MyDrive/ComfyUI/models/`(checkpoints / loras / vae / clip / text_encoders / diffusion_models)
- 输入图读取 `MyDrive/ComfyUI/input/`,输出直接写到 `MyDrive/ComfyUI/output/`(会话结束也不丢)
- 空闲 30 分钟自动释放 GPU

从上往下依次运行。第 5 格会打印访问地址。

## 1. GPU 与 Drive

In [ ]:
import os, sys, subprocess, json, re, time, threading
from pathlib import Path
subprocess.run(['nvidia-smi'], check=True)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_COMFY = Path('/content/drive/MyDrive/ComfyUI')
MODELS_DIR = DRIVE_COMFY / 'models'
INPUT_DIR  = DRIVE_COMFY / 'input'
OUTPUT_DIR = DRIVE_COMFY / 'output'
for d in (INPUT_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)
assert MODELS_DIR.is_dir(), 'MyDrive/ComfyUI/models not found'
print({p.name: len(list(p.iterdir())) for p in MODELS_DIR.iterdir() if p.is_dir()})

## 2. 安装 ComfyUI(最新版)和 ComfyUI-Manager
需要额外自定义节点(SAM3、Impact Pack 等)时,在网页里用 Manager 安装即可;自定义节点存在 Colab 本地盘,新会话需要重新装。

In [ ]:
COMFY = Path('/content/ComfyUI')
if not COMFY.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI.git', str(COMFY)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')], check=True)
manager = COMFY / 'custom_nodes' / 'ComfyUI-Manager'
if not manager.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Comfy-Org/ComfyUI-Manager.git', str(manager)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(manager / 'requirements.txt')], check=False)
print('ready')

## 3. 模型路径指向 Drive

In [ ]:
yaml = f'''
drive:
  base_path: {MODELS_DIR}
  checkpoints: checkpoints
  loras: loras
  vae: vae
  clip: clip
  text_encoders: text_encoders
  diffusion_models: diffusion_models
'''
EXTRA = Path('/content/extra_model_paths.yaml')
EXTRA.write_text(yaml)
print(yaml)

## 4. 启动 ComfyUI

In [ ]:
import requests
if 'comfy_proc' in globals() and comfy_proc.poll() is None:
    raise RuntimeError('ComfyUI is already running.')
comfy_log = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen([
    sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188',
    '--extra-model-paths-config', str(EXTRA),
    '--input-directory', str(INPUT_DIR), '--output-directory', str(OUTPUT_DIR),
    '--preview-method', 'auto',
], cwd=COMFY, stdout=comfy_log, stderr=subprocess.STDOUT)
for _ in range(120):
    if comfy_proc.poll() is not None:
        print(Path('/content/comfyui.log').read_text()[-5000:])
        raise RuntimeError('ComfyUI exited; see log above.')
    try:
        if requests.get('http://127.0.0.1:8188/system_stats', timeout=3).status_code == 200:
            print('ComfyUI is up on port 8188')
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise TimeoutError('ComfyUI did not start; check /content/comfyui.log')

## 5. 打开网页
**注意**:ComfyUI 自身没有登录。下面的 Cloudflare 地址是随机的公开链接,**任何拿到链接的人都能用你的 GPU 并通过 Manager 装节点**,不要分享、不要截图带出地址。
想更安全可以把 `TUNNEL` 改成 `'colab'`,走 Colab 自带的代理(需要登录你的 Google 账号),但个别情况下 WebSocket 进度可能不稳定。

In [ ]:
TUNNEL = 'cloudflared'   # 'cloudflared' 或 'colab'

if TUNNEL == 'colab':
    from google.colab import output
    output.serve_kernel_port_as_window(8188, path='/')
else:
    cf = Path('/content/cloudflared')
    if not cf.exists():
        url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
        with requests.get(url, stream=True, timeout=(30, 120)) as r:
            r.raise_for_status()
            with cf.open('wb') as fh:
                for chunk in r.iter_content(1 << 20):
                    fh.write(chunk)
        cf.chmod(0o755)
    if 'tunnel' not in globals() or tunnel.poll() is not None:
        tunnel = subprocess.Popen([str(cf), 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
                                  stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/tunnel.log').read_text())
        if m:
            print('ComfyUI:', m.group(0))
            break
        time.sleep(1)
    else:
        raise TimeoutError('Tunnel URL not ready; see /content/tunnel.log')

## 6. 空闲自动释放 GPU
队列里有任务、或有新任务完成,都会重置计时。只在网页里改工作流不算活动。修改 `IDLE_MINUTES` 后重新运行此格。

In [ ]:
from google.colab import runtime
IDLE_MINUTES = 30
def _activity():
    q = requests.get('http://127.0.0.1:8188/queue', timeout=5).json()
    h = requests.get('http://127.0.0.1:8188/history?max_items=1', timeout=5).json()
    return bool(q.get('queue_running') or q.get('queue_pending')), (next(iter(h), None))

def _watch(stop, limit):
    last_active, last_hist = time.monotonic(), None
    while not stop.wait(10):
        try:
            busy, hist = _activity()
        except requests.RequestException:
            last_active = time.monotonic()      # cannot confirm idle
            continue
        if busy or hist != last_hist:
            last_active = time.monotonic()
        last_hist = hist
        if time.monotonic() - last_active >= limit:
            print('Idle for', IDLE_MINUTES, 'minutes; releasing runtime.')
            runtime.unassign()
            return

if 'idle_stop' in globals():
    idle_stop.set(); time.sleep(11)
idle_stop = threading.Event()
threading.Thread(target=_watch, args=(idle_stop, IDLE_MINUTES * 60), daemon=True).start()
print('Auto release enabled:', IDLE_MINUTES, 'min idle')

## 手动释放 / 关闭定时器 / 看日志
```python
from google.colab import runtime; runtime.unassign()   # 立即释放
idle_stop.set()                                         # 只关定时器
!tail -n 80 /content/comfyui.log                        # ComfyUI 日志
```